In [ ]:
import re
import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# ==========================================
# 1. FUNÇÃO DE PRÉ-PROCESSAMENTO (MASKING)
# ==========================================
def pre_processar_lai(texto):
    if not isinstance(texto, str):
        return ""

    texto = texto.lower()

    # Normalização de estrutura (Substituição por Tokens)
    texto = re.sub(r'https?://\S+|www\.\S+', ' tokenurl ', texto)
    texto = re.sub(r'(lei|decreto|súmula|sumula)\s*(nº|n°|n\.º)?\s*[\d\.\/-]+', ' tokenlei ', texto)
    texto = re.sub(r'\b\d{5}\.\d{6}/\d{4}-\d{2}\b', ' tokenprocesso ', texto)
    texto = re.sub(r'\b\d+[\d\.-]*\b', ' tokennumero ', texto)

    # Limpeza de pontuação e espaços
    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto



In [ ]:
# ==========================================
# 2. CARREGAMENTO DO ARQUIVO XLSX
# ==========================================
caminho_arquivo = '/train.xlsx'

# Carrega a planilha em um DataFrame do pandas
df_treino = pd.read_excel(caminho_arquivo)

# Exibe as primeiras linhas e os nomes das colunas para verificação
print("Colunas encontradas no arquivo:", df_treino.columns.tolist())
print(f"Total de registros carregados: {len(df_treino)}\n")

Colunas encontradas no arquivo: ['resp_text', 'clarity']
Total de registros carregados: 20092



In [ ]:
# ==========================================
# 3. TRANSFORMAÇÃO E PREPARAÇÃO DO DATASET
# ==========================================
# Ajuste aqui os nomes das colunas de acordo com o seu XLSX
COLUNA_TEXTO = 'resp_text'  # Exemplo baseado na amostra anterior
COLUNA_CLASSE = 'clarity'           # Exemplo: 'c1', 'c2', 'c5', etc.

# A. Limpeza de valores nulos/ausentes
df_treino = df_treino.dropna(subset=[COLUNA_TEXTO, COLUNA_CLASSE])

# B. Aplicação do pré-processamento na coluna de texto
print("Aplicando pré-processamento nos textos...")
df_treino['texto_limpo'] = df_treino[COLUNA_TEXTO].apply(pre_processar_lai)

# C. Separação nas variáveis X_train (features) e y_train (rótulos)
X_train = df_treino['texto_limpo']
y_train = df_treino[COLUNA_CLASSE]

print("Amostra do texto após limpeza:\n", X_train.iloc[0][:200], "...\n")

Aplicando pré-processamento nos textos...
Amostra do texto após limpeza:
 prezado a senhor a esclarecemos que o serviço de informação ao cidadão sic não trata de sugestão elogio solicitação reclamação e denúncia relacionados a objetos postados no brasil ou no exterior cumpr ...



In [ ]:
# ==========================================
# 4. VALIDAÇÃO CRUZADA E PIPELINE
# ==========================================
# StratifiedKFold conforme configurado no seu setup inicial
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline de Execução (TF-IDF + SVM Linear)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(dual=False, random_state=42, max_iter=2000))  # <--- dual=False adicionado
])


# ==========================================
# 3. GRADE DE PARÂMETROS
# ==========================================
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5, 10],
    'tfidf__max_df': [0.50, 0.70, 0.85],
    'tfidf__sublinear_tf': [True, False],
    'clf__C': [0.1, 1, 10]
}


# ==========================================
# 3. GRID SEARCH COM ACURÁCIA
# ==========================================
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',  # <--- Seleção baseada em Acurácia
    n_jobs=-1,
    verbose=1
)



In [ ]:
# Executa o ajuste no conjunto de treino
# print("Iniciando o Grid Search com StratifiedKFold...")
# grid_search.fit(X_train, y_train)


# ==========================================
# 4. EXIBIÇÃO EXCLUSIVA DO MELHOR MODELO
# ==========================================
# Descobre o índice do melhor modelo já treinado
best_idx = grid_search.best_index_
results = grid_search.cv_results_

print("\n==================================================")
print("          RESULTADO DO MELHOR MODELO              ")
print("==================================================")

# Imprime todas as métricas médias encontradas no seu cv_results_ atual
for key in results.keys():
    if key.startswith('mean_test_'):
        nome_metrica = key.replace('mean_test_', '')
        print(f"  • {nome_metrica.capitalize()}: {results[key][best_idx]:.4f}")

print("\n⚙️ MELHORES HIPERPARÂMETROS SELECIONADOS:")
for param, val in grid_search.best_params_.items():
    print(f"  • {param}: {val}")
print("==================================================")
# O modelo final ajustado com a melhor combinação fica disponível em:
melhor_modelo = grid_search.best_estimator_


          RESULTADO DO MELHOR MODELO              
  • Score: 0.4610

⚙️ MELHORES HIPERPARÂMETROS SELECIONADOS:
  • clf__C: 0.1
  • tfidf__max_df: 0.7
  • tfidf__min_df: 2
  • tfidf__ngram_range: (1, 2)
  • tfidf__sublinear_tf: True


# Expansão do Grid Search pro SVM

In [ ]:



# ==========================================
# VALIDAÇÃO CRUZADA
# ==========================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ==========================================
# PIPELINE
# ==========================================

pipeline = Pipeline([
    (
        'tfidf',
        TfidfVectorizer()
    ),
    (
        'clf',
        LinearSVC(
            dual=False,
            random_state=42,
            max_iter=5000
        )
    )
])


# ==========================================
# GRID SEARCH
# ==========================================
#
# VERSÃO REDUZIDA (recomendada)
# ~128 combinações
#
# Se encontrarem melhora relevante,
# depois expandam para um grid maior.
#
# ==========================================

# ==========================================
# GRID SEARCH
# ==========================================

param_grid = {

    # TF-IDF

    'tfidf__ngram_range': [
        (1, 1),
        (1, 2),
        (1, 3)
    ],

    'tfidf__min_df': [
        2,
        5,
        10
    ],

    'tfidf__max_df': [
        0.50,
        0.70,
        0.85
    ],

    'tfidf__sublinear_tf': [
        True,
        False
    ],

    'tfidf__max_features': [
        10000,
        20000,
        None
    ],

    # SVM

    'clf__C': [
        0.01,
        0.1,
        1,
        10,
        100
    ],

    'clf__class_weight': [
        None,
        'balanced'
    ]
}


# ==========================================
# GRID SEARCH
# ==========================================

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)


# ==========================================
# TREINAMENTO
# ==========================================

print("Iniciando Grid Search...\n")

grid_search.fit(
    X_train,
    y_train
)


# ==========================================
# MELHOR MODELO
# ==========================================

print("\n" + "="*60)
print("MELHOR MODELO ENCONTRADO")
print("="*60)

print(
    f"\nAccuracy média (CV): "
    f"{grid_search.best_score_:.4f}"
)

print("\nMelhores hiperparâmetros:")

for parametro, valor in grid_search.best_params_.items():
    print(f"{parametro}: {valor}")


# ==========================================
# RESULTADOS COMPLETOS
# ==========================================

results = pd.DataFrame(
    grid_search.cv_results_
)

results = results.sort_values(
    by='mean_test_score',
    ascending=False
)


# ==========================================
# TOP 10 CONFIGURAÇÕES
# ==========================================

print("\n" + "="*60)
print("TOP 10 CONFIGURAÇÕES")
print("="*60)

print(
    results[
        [
            'mean_test_score',
            'std_test_score',
            'mean_train_score',
            'params'
        ]
    ].head(10)
)


# ==========================================
# ESTATÍSTICAS GERAIS
# ==========================================

print("\n" + "="*60)
print("ESTATÍSTICAS GERAIS")
print("="*60)

print(
    f"Maior Accuracy: "
    f"{results['mean_test_score'].max():.4f}"
)

print(
    f"Menor Accuracy: "
    f"{results['mean_test_score'].min():.4f}"
)

print(
    f"Média das Accuracies: "
    f"{results['mean_test_score'].mean():.4f}"
)

print(
    f"Desvio Padrão: "
    f"{results['mean_test_score'].std():.4f}"
)


# ==========================================
# MELHOR CONFIGURAÇÃO
# ==========================================

print("\n" + "="*60)
print("MELHOR CONFIGURAÇÃO")
print("="*60)

print(
    results[
        [
            'mean_test_score',
            'std_test_score',
            'params'
        ]
    ].iloc[0]
)


# ==========================================
# PIOR CONFIGURAÇÃO
# ==========================================

print("\n" + "="*60)
print("PIOR CONFIGURAÇÃO")
print("="*60)

print(
    results[
        [
            'mean_test_score',
            'std_test_score',
            'params'
        ]
    ].iloc[-1]
)


# ==========================================
# EXPORTAR RESULTADOS (OPCIONAL)
# ==========================================

results.to_csv(
    "resultados_gridsearch_svm.csv",
    index=False
)

print(
    "\nResultados completos salvos em:"
    "\nresultados_gridsearch_svm.csv"
)

Iniciando Grid Search...

Fitting 5 folds for each of 1620 candidates, totalling 8100 fits


# Modelo de embeddings pre treinados com svm

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# ==========================================
# 1. CARREGAMENTO DAS REPRESENTAÇÕES EMBEDDING
# ==========================================
def gerar_embeddings(textos, nome_modelo):
    print(f"\nCarregando modelo de Embedding: {nome_modelo}...")
    model = SentenceTransformer(nome_modelo)

    print("Gerando vetores densos para o dataset...")
    # Convert_to_numpy garante saída no formato correto para o Scikit-Learn
    embeddings = model.encode(textos.tolist(), show_progress_bar=True, convert_to_numpy=True)
    return embeddings

# Nota: Assumindo que X_train e y_train já estão no seu ambiente.
# Se X_train contiver os textos pré-processados (com masking ou texto limpo), passe-os diretamente:

modelos_embedding = {
    'distilUSE': 'sentence-transformers/distiluse-base-multilingual-cased-v2',
    'MiniLM': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
}

# Dicionário para armazenar os resultados dos dois experimentos
resultados_experimentos = {}

# ==========================================
# 2. CONFIGURAÇÃO DA VALIDAÇÃO CRUZADA E GRID SEARCH
# ==========================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grade de otimização do SVM sobre espaço vetorial denso
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100]
}

scoring = {
    'acuracia': 'accuracy',
    'f1': 'f1_macro',
    'precisao': 'precision_macro',
    'revocacao': 'recall_macro'
}


# ==========================================
# 3. EXECUÇÃO DOS EXPERIMENTOS
# ==========================================
for nome, modelo_path in modelos_embedding.items():
    print(f"\n==================================================")
    print(f"       INICIANDO EXPERIMENTO COM {nome.upper()}")
    print(f"==================================================")

    # Extração dos Embeddings
    X_train_emb = gerar_embeddings(X_train, modelo_path)

    # Instanciação do Grid Search no SVM
    # Obs: Com embeddings densos, dual=False com LinearSVC é mais rápido quando n_samples > n_features
    svm = LinearSVC(dual=False, random_state=42, max_iter=3000)

    grid_search = GridSearchCV(
        estimator=svm,
        param_grid=param_grid,
        cv=cv,
        scoring=scoring,
        refit='acuracia',  # Otimiza para Acurácia
        n_jobs=-1,
        verbose=1
    )

    print(f"Ajustando SVM com Grid Search sobre os embeddings do {nome}...")
    grid_search.fit(X_train_emb, y_train)

    # Armazena os resultados
    best_idx = grid_search.best_index_
    res = grid_search.cv_results_

    resultados_experimentos[nome] = {
        'best_estimator': grid_search.best_estimator_,
        'best_params': grid_search.best_params_,
        'acuracia': res['mean_test_acuracia'][best_idx],
        'f1': res['mean_test_f1'][best_idx],
        'precisao': res['mean_test_precisao'][best_idx],
        'revocacao': res['mean_test_revocacao'][best_idx]
    }


# ==========================================
# 4. COMPARAÇÃO FINAL DOS MODELOS
# ==========================================
print("\n" + "="*60)
print("             TABELA COMPARATIVA DE RESULTADOS            ")
print("="*60)

df_resultados = pd.DataFrame(resultados_experimentos).T
df_resultados_exibicao = df_resultados[['acuracia', 'f1', 'precisao', 'revocacao']].copy()
df_resultados_exibicao.columns = ['Acurácia', 'F1-Score (Macro)', 'Precisão (Macro)', 'Revocação (Macro)']

print(df_resultados_exibicao.to_string())

print("\n⚙️ MELHORES PARÂMETROS ENCONTRADOS:")
for nome in modelos_embedding.keys():
    print(f"  • {nome}: {resultados_experimentos[nome]['best_params']}")
print("="*60)


       INICIANDO EXPERIMENTO COM DISTILUSE

Carregando modelo de Embedding: sentence-transformers/distiluse-base-multilingual-cased-v2...


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.46k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Gerando vetores densos para o dataset...


Batches:   0%|          | 0/628 [00:00<?, ?it/s]

Ajustando SVM com Grid Search sobre os embeddings do MiniLM...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

             TABELA COMPARATIVA DE RESULTADOS            
           Acurácia F1-Score (Macro) Precisão (Macro) Revocação (Macro)
distilUSE  0.427683         0.421972          0.42372          0.427759
MiniLM     0.430072         0.424009         0.425975            0.4303

⚙️ MELHORES PARÂMETROS ENCONTRADOS:
  • distilUSE: {'C': 0.1}
  • MiniLM: {'C': 0.01}
